# Unsupervised machine learning pipeline

Explore patterns in Disque 100 human-trafficking records without a target variable. This notebook covers **data collection, data preparation, feature engineering, model training, and model evaluation**.

The unit of analysis is a source record, not a unique person or report. Multiple records can share a report hash; only exact duplicate records are removed. Clusters describe similarity in reporting context, not confirmed trafficking, culpability, or individual risk. Reporting bias and repeated reports can dominate apparent patterns.

## Setup and reproducibility
Install project dependencies from the repository root with `uv sync`, select the project's `.venv` kernel, and run all cells in order. Input stays local; no raw records or identifiers are displayed or exported. Saved notebook outputs are intentionally empty. Clear outputs before sharing, and review even aggregate results for disclosure risk.


In [ ]:
from pathlib import Path
import hashlib
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    adjusted_rand_score, calinski_harabasz_score,
    davies_bouldin_score, silhouette_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
MAX_ROWS = 5000
MAX_COMPONENTS = 20
SILHOUETTE_SAMPLE_SIZE = 1000
K_VALUES = range(2, 7)
MIN_PROFILE_SIZE = 10
DATA_FILENAME = 'trafico-pessoas-2021-2025.csv'

print({'python': platform.python_version(), 'pandas': pd.__version__,
       'numpy': np.__version__, 'scikit-learn': sklearn.__version__})


## 1. Data collection

Use the existing semicolon-separated UTF-8 dataset in `data/`. Run from the repository root or `notebooks/`; otherwise set `PROJECT_ROOT` explicitly. Do not concatenate the related violations file without checking overlap and the unit of analysis. Record the input checksum to identify the exact dataset used.


In [ ]:
cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd.parent]
PROJECT_ROOT = next(
    (root for root in root_candidates if (root / 'pyproject.toml').is_file()
     and (root / 'data' / DATA_FILENAME).is_file()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory; check data/ input.')

DATA_PATH = PROJECT_ROOT / 'data' / DATA_FILENAME
raw = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8-sig', dtype='string')
if raw.empty:
    raise ValueError('The input dataset contains no records.')
input_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print({'file': DATA_PATH.name, 'rows': len(raw), 'columns': raw.shape[1],
       'sha256': input_sha256})


## 2. Data preparation

Remove exported index columns, trim whitespace, normalize known missing-value markers, and remove exact duplicates. Keep different records sharing a report identifier: selecting one arbitrarily could discard distinct victims or violations. Sampling, if needed, is reproducible and caps hierarchical clustering's quadratic memory cost. This is not a population-representative sample.

Use a conservative feature allowlist. Report hashes, municipalities, free text, and victim/suspect demographic attributes are excluded. The relationship field still carries sensitive context and needs governance review before operational use. Missing fields raise an error rather than silently changing the model inputs.


In [ ]:
CATEGORICAL_FEATURES = [
    'canal_de_atendimento', 'denuncia_emergencial', 'cenario_da_violacao',
    'uf', 'frequencia', 'inicio_das_violacoes', 'relacao_vitima_suspeito',
]
DATE_COLUMN = 'data_de_cadastro'
required_columns = CATEGORICAL_FEATURES + [DATE_COLUMN, 'hash']
missing_columns = sorted(set(required_columns) - set(raw.columns))
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

df = raw.loc[:, ~raw.columns.str.startswith('Unnamed:')].copy()
missing_markers = ['', 'null', 'nan', 'none', 'não informado', 'nao informado',
                   'não se aplica', 'nao se aplica']
for column in df.columns:
    values = df[column].str.strip()
    df[column] = values.mask(values.str.casefold().isin(missing_markers))
exact_duplicates = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
repeated_report_rows = int(df['hash'].dropna().duplicated().sum())
rows_before_sampling = len(df)
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=RANDOM_STATE).sort_index()
df = df.reset_index(drop=True)
if len(df) < 4:
    raise ValueError('At least four prepared records are needed for clustering evaluation.')

print({'exact_duplicates_removed': exact_duplicates,
       'additional_rows_sharing_a_report_hash': repeated_report_rows,
       'rows_before_sampling': rows_before_sampling, 'analysis_rows': len(df)})
display(df[CATEGORICAL_FEATURES].isna().mean().rename('missing_fraction').to_frame())


## 3. Feature engineering

Encode registration month cyclically; registration is not the date of the violation. The year is intentionally excluded to reduce separation driven by reporting-system changes. Invalid dates are counted and imputed, not silently discarded.

A scikit-learn preprocessing pipeline imputes missing values, groups rare categories, one-hot encodes nominal fields, and scales numerical fields. Truncated SVD produces a bounded dense representation for both models; L2 normalization emphasizes relative profiles. Distances and metrics refer to this representation, not original categories. Inspect sensitivity to feature selection, rare-category thresholds, and component count before interpreting clusters.

All available analysis rows are used for exploratory fitting. This is **not** held-out validation or a predictive evaluation. Any later supervised workflow must split by report/person (and possibly time) before fitting its preprocessing.


In [ ]:
dates = pd.to_datetime(df[DATE_COLUMN], format='mixed', errors='coerce')
invalid_dates = int((df[DATE_COLUMN].notna() & dates.isna()).sum())
print({'invalid_registration_dates': invalid_dates, 'missing_registration_dates': int(dates.isna().sum())})
if dates.notna().sum() == 0:
    raise ValueError('No registration dates could be parsed; check the input date format.')

features = df[CATEGORICAL_FEATURES].copy()
month_angle = 2 * np.pi * (dates.dt.month - 1) / 12
features['registration_month_sin'] = np.sin(month_angle)
features['registration_month_cos'] = np.cos(month_angle)
NUMERIC_FEATURES = ['registration_month_sin', 'registration_month_cos']
all_missing = features.columns[features.isna().all()].tolist()
if all_missing:
    raise ValueError(f'Features contain no observed values: {all_missing}')
features[CATEGORICAL_FEATURES] = features[CATEGORICAL_FEATURES].astype(object).where(
    features[CATEGORICAL_FEATURES].notna(), np.nan
)

preprocessor = ColumnTransformer([
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encode', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                 min_frequency=0.01, max_categories=30)),
    ]), CATEGORICAL_FEATURES),
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), NUMERIC_FEATURES),
], sparse_threshold=1.0)
encoded = preprocessor.fit_transform(features)
n_components = min(MAX_COMPONENTS, encoded.shape[0] - 1, encoded.shape[1] - 1)
if n_components < 2:
    raise ValueError('Insufficient feature dimensions for the exploratory representation.')
representation = Pipeline([
    ('svd', TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)),
    ('normalize', Normalizer()),
])
X = representation.fit_transform(encoded)
if not np.isfinite(X).all():
    raise ValueError('The feature representation contains non-finite values.')
unique_profiles = np.unique(X, axis=0).shape[0]
if unique_profiles < 3:
    raise ValueError('At least three distinct feature profiles are needed for this comparison.')
print({'encoded_shape': encoded.shape, 'model_shape': X.shape,
       'svd_explained_variance': float(representation.named_steps['svd'].explained_variance_ratio_.sum())})


## 4. Model training

Compare **K-Means** and **agglomerative clustering (Ward linkage)** for several cluster counts using the same representation. Neither model needs a target variable. K-Means uses multiple initializations and a fixed seed; Ward is deterministic for fixed row order. The default 5,000-record cap limits, but does not eliminate, hierarchical memory costs. No model files or record-level assignments are written to disk.


In [ ]:
models = {}
assignments = {}
candidate_ks = [k for k in K_VALUES if k < len(X) and k <= unique_profiles]
if not candidate_ks:
    raise ValueError('No valid cluster counts for the prepared dataset.')
for k in candidate_ks:
    candidates = {
        f'kmeans_k{k}': KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE),
        f'ward_k{k}': AgglomerativeClustering(n_clusters=k, linkage='ward'),
    }
    for name, model in candidates.items():
        labels = model.fit_predict(X)
        models[name] = model
        assignments[name] = labels
print(f'Trained {len(models)} candidate models without a target variable.')


## 5. Model evaluation

Without labels, accuracy, precision, recall, and ROC-AUC are not applicable. Compare internal measures: **silhouette** and **Calinski-Harabasz** (higher is better), **Davies-Bouldin** (lower is better), cluster sizes, and sensitivity to random initialization. Silhouette uses at most 1,000 fixed randomly selected records; the other indices use all analysis records. Degenerate candidates and unscorable silhouette samples are reported explicitly.

The highest silhouette is only a provisional ranking on the fitted sample, not proof of real-world validity. Seed agreement measures optimizer sensitivity, not resampling stability or generalization. Review cluster meaning with domain experts and check stability across report-grouped samples, time windows, and alternative representations before choosing a model.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
evaluation_indices = np.sort(rng.choice(len(X), size=min(SILHOUETTE_SAMPLE_SIZE, len(X)), replace=False))
metric_rows = []
for name, labels in assignments.items():
    counts = pd.Series(labels).value_counts()
    n_labels = len(counts)
    if not 2 <= n_labels < len(X):
        print(f'Skipping {name}: {n_labels} distinct clusters cannot be scored.')
        continue
    sampled_labels = labels[evaluation_indices]
    sample_is_valid = 2 <= len(np.unique(sampled_labels)) < len(sampled_labels)
    if not sample_is_valid:
        print(f'{name}: silhouette sample has an invalid cluster count; score is unavailable.')
    metric_rows.append({
        'model': name, 'clusters': n_labels,
        'silhouette': silhouette_score(X[evaluation_indices], sampled_labels) if sample_is_valid else np.nan,
        'calinski_harabasz': calinski_harabasz_score(X, labels),
        'davies_bouldin': davies_bouldin_score(X, labels),
        'smallest_cluster': int(counts.min()), 'largest_cluster_fraction': float(counts.max() / len(X)),
    })
if not metric_rows:
    raise ValueError('No candidate produced a valid clustering.')
results = pd.DataFrame(metric_rows).set_index('model').sort_values('silhouette', ascending=False)
display(results.round(4))
ranked = results.dropna(subset=['silhouette'])
if ranked.empty:
    raise ValueError('No valid silhouette scores; increase the evaluation sample size.')
best_name = ranked.index[0]
best_labels = assignments[best_name]
print(f'Provisional silhouette leader: {best_name}; domain review is still required.')


In [ ]:
stability_rows = []
for name, model in models.items():
    if isinstance(model, KMeans):
        for seed in [7, 21, 84]:
            alternate = KMeans(n_clusters=model.n_clusters, n_init=20, random_state=seed).fit_predict(X)
            stability_rows.append({'model': name, 'seed': seed,
                                   'adjusted_rand_agreement': adjusted_rand_score(assignments[name], alternate)})
stability = pd.DataFrame(stability_rows)
display(stability.groupby('model')['adjusted_rand_agreement'].agg(['min', 'mean']).round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
results['silhouette'].plot.bar(ax=axes[0], title='Silhouette by candidate (higher is better)')
pd.Series(best_labels).value_counts().sort_index().plot.bar(
    ax=axes[1], title=f'Record counts: {best_name}'
)
axes[0].set_ylabel('Silhouette')
axes[1].set_xlabel('Cluster (arbitrary identifier)')
axes[1].set_ylabel('Records')
plt.tight_layout()
plt.show()


In [ ]:
profile_data = features[CATEGORICAL_FEATURES].copy()
profile_data['cluster'] = best_labels
profile_rows = []
for cluster, group in profile_data.groupby('cluster'):
    if len(group) < MIN_PROFILE_SIZE:
        print(f'Cluster {cluster}: profile suppressed (fewer than {MIN_PROFILE_SIZE} records).')
        continue
    for column in CATEGORICAL_FEATURES:
        counts = group[column].fillna('missing').value_counts()
        if counts.iloc[0] < MIN_PROFILE_SIZE:
            continue
        profile_rows.append({'cluster': cluster, 'feature': column,
                             'most_common_value': counts.index[0],
                             'share': float(counts.iloc[0] / len(group))})
display(pd.DataFrame(profile_rows, columns=['cluster', 'feature', 'most_common_value', 'share']))


### Interpretation and next steps

- Record the checksum, package versions, feature allowlist, sample size, SVD variance, metrics, and seed agreement when documenting an experiment.
- Check whether clusters mainly reflect reporting channels, missingness, or duplicate report contexts rather than substantive differences. The same report can contribute several rows and disproportionately influence fitting and metrics.
- Review alternative feature sets and rare-category/SVD settings. One-hot Euclidean distances and SVD can distort mixed categorical profiles; compare an appropriate categorical-distance method if needed.
- Small-cell suppression here is only a basic precaution, not a privacy guarantee. Do not publish raw assignments or sensitive profiles without disclosure review.
- Document expert interpretation and report-grouped/temporal stability before adopting any cluster count. Do not automatically name clusters as risk groups.
- Once a defensible target is selected, define its provenance and leakage exclusions, then create a separate supervised pipeline with report-grouped/time-aware train-validation-test splits. Cluster IDs must not be treated as ground-truth targets.
